# P-Tuning v1 交互教学

配套 lecture：[`../lectures/03-p-tuning.md`](../lectures/03-p-tuning.md)  
配套论文：[`../papers/03-p-tuning-2021.pdf`](../papers/03-p-tuning-2021.pdf)

本 notebook 演示：
1. 环境检查
2. 手写 minimal 版（含 Bi-LSTM + MLP）
3. peft 调包版
4. 数值一致性验证（弱一致）
5. 参数量拆解：embedding / LSTM / MLP
6. 思考题

In [ ]:
import sys
import torch, transformers, peft
print(f'Python: {sys.version.split()[0]}')
print(f'torch:        {torch.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'peft:         {peft.__version__}')

## 2. 手写 minimal 版

公式 (2): $(h_0, ..., h_{p-1}) = \mathrm{MLP}_\phi(\mathrm{BiLSTM}_\phi(u_0, ..., u_{p-1}))$

In [ ]:
from pathlib import Path
src_dir = (Path.cwd().parent / 'src').resolve()
sys.path.insert(0, str(src_dir))

from p_tuning_minimal import PTuningGPT2, PromptEncoder
from common import print_param_summary

torch.manual_seed(42)
model = PTuningGPT2(prompt_length=10, encoder_hidden=256)
print_param_summary(model, 'minimal (p=10, h=256)')

In [ ]:
# 参数拆解
n_embed = model.encoder.embed.weight.numel()
n_lstm = sum(p.numel() for p in model.encoder.lstm.parameters())
n_mlp = sum(p.numel() for p in model.encoder.mlp.parameters())
print(f'embedding (u_i):  {n_embed:>10,}  ← 公式 (2) 的原始 u')
print(f'Bi-LSTM:          {n_lstm:>10,}  ← 公式 (2) 的 BiLSTM_φ')
print(f'MLP head:         {n_mlp:>10,}  ← 公式 (2) 的 MLP_φ')
print(f'-------------------------------------')
print(f'合计:             {n_embed + n_lstm + n_mlp:>10,}')
print(f'\n注意：LSTM 占绝大部分（{100*n_lstm/(n_embed+n_lstm+n_mlp):.1f}%）')

## 3. PromptEncoder 内部行为

调用 `encoder()` 单独看：输入 $\{u_i\}$ → BiLSTM → MLP → 输出 $\{h_i\}$。

In [ ]:
with torch.no_grad():
    h = model.encoder()
print(f'输出 prompt embedding 形状: {tuple(h.shape)}  (期望 (10, 768))')
print(f'第 0 个 pseudo token 的前 5 个值: {h[0, :5].tolist()}')

## 4. peft 调包版

In [ ]:
from p_tuning_peft import build_peft_model
torch.manual_seed(42)
peft_model = build_peft_model(prompt_length=10)
print_param_summary(peft_model, 'peft (p=10, h=256)')

## 5. 数值一致性验证（弱一致）

In [ ]:
tests_dir = (Path.cwd().parent / 'src' / 'tests').resolve()
sys.path.insert(0, str(tests_dir))
from test_p_tuning_consistency import test_shape_and_param_order
test_shape_and_param_order()

## 6. 思考题

1. **LSTM 参数量公式**：双向 LSTM 2 层、input=768、hidden=256，参数量怎么算的？请验证。
2. **改 hidden 维度**：把 `encoder_hidden` 从 256 改成 128，参数量变多少？预测后验证。
3. **MLP vs LSTM**：peft 通过 `PromptEncoderReparameterizationType.MLP` 提供 MLP 模式，参数量会怎么变？
4. **prompt_length 影响**：$p$ 从 10 改到 100，**LSTM 参数变吗**？为什么？（提示：LSTM 参数与序列长度无关，只与 hidden、input 有关。）
5. **与 Prompt Tuning 对比**：本 notebook 4.3M 参数；[`02-prompt-tuning.ipynb`](02-prompt-tuning.ipynb) 7.7K 参数。在 toy 任务上谁更容易过拟合？

In [ ]:
# 提示：思考题 4
torch.manual_seed(0)
model_p100 = PTuningGPT2(prompt_length=100, encoder_hidden=256)
n_lstm_p100 = sum(p.numel() for p in model_p100.encoder.lstm.parameters())
print(f'p=100 时 LSTM 参数: {n_lstm_p100:,}')
print(f'p=10  时 LSTM 参数: 3,678,208')
print(f'结论: LSTM 参数 {("不变" if n_lstm_p100 == 3678208 else "变了")}！')
print(f'（embedding 部分 (p, d) 从 7,680 变到 76,800，但 LSTM 与 p 无关）')